
# AI Engineering — Week 3 Lab
## Data Engineering for AI: Pipelines, Lineage, and Workflows

**Project:** UTM FAI FYP & Supervisor Matching Assistant  
**Course:** SAIA 3113 — AI Engineering  
**Platform:** Google Colab  
**Notebook:** `week3_pipeline.ipynb`

### Selected Data Sources

1. **UTM FAI Lecturer Profiles (TXT)** — Lecturer names, departments, research expertise and official source URLs.
2. **UTM FAI FYP Handbook (PDF)** — Official FYP rules, requirements and procedures.

# Part A

## A1: Requirements-to-data mapping



The selected datasets are based on the requirements and V1 architecture established in Topic 2.

| Requirement | Needed Information | Selected Source | Justification / Constraint |
|---|---|---|---|
| FR1 — Supervisor Matching | Lecturer names, departments and research expertise | UTM FAI Lecturer Profiles (TXT) | Supports matching FYP ideas to relevant research areas. Missing expertise may limit recommendations. |
| FR2 — FYP Policy Q&A | Official FYP rules, procedures and requirements | UTM FAI FYP Handbook (PDF) | Provides an authoritative document for answering FYP-related questions. |
| FR3 — Scope Refinement | Research expertise and relevant FYP guidance | Lecturer TXT and FYP Handbook PDF | Helps provide suggestions grounded in available faculty expertise and official guidance. |
| FR5 — Citations | Source URL, filename and page information where available | Both datasets | Supports traceable and verifiable responses. |
| NFR4 — Source Updates | Source identity, collection date and file version | Dataset metadata | Supports reproducibility and future updates. |

### Selected Data Preparation Pathway

**Pathway A — PDF/TXT Documents**

Both sources contain textual information. Therefore, the project will use document parsing, text cleaning, chunking, validation and metadata preservation.

The lecturer TXT contains manually reviewed records, while the FYP Handbook PDF contains official institutional guidance.

### Data Limitations

- Some lecturer profiles do not provide research expertise.
- The lecturer profile dataset was compiled from official UTM Faculty of Artificial Intelligence (FAI) department webpages. The collected information was manually reviewed, corrected, and organized into a structured TXT file for further processing.
- Lecturer research expertise does not confirm supervisor availability.
- PDF extraction may introduce formatting or reading-order issues.


## A2: Set up the Colab workspace

### A2. 1 Workspace setup

In [1]:
# ============================================================
# CELL 1 — WORKSPACE SETUP
# Part A2: Initialize project directories
# ============================================================

from pathlib import Path

# Define project directories
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create directories
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for later lab activities
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

# Verify workspace
print("Workspace initialized successfully!")
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Workspace initialized successfully!
Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


### A2. 2 Verify Uploaded Inputs

In [6]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
fai_all_lecturer_profiles.txt bytes: 34172
utm_fai_fyp_guidelines.pdf.pdf bytes: 89523865
Upload verified. Select your relevant input(s) for Step A3.


### A2. 3 Check Required Dataset Types

In [3]:
# ============================================================
# CELL 3A — CHECK REQUIRED DATASET TYPES
# Part A2: Confirm both project sources are present
# ============================================================

txt_files = [f for f in files if f.suffix.lower() == ".txt"]
pdf_files = [f for f in files if f.suffix.lower() == ".pdf"]

print("TXT files found:", len(txt_files))
print("PDF files found:", len(pdf_files))

assert len(txt_files) == 1, "Expected one lecturer TXT file."
assert len(pdf_files) == 1, "Expected one FYP Handbook PDF."

print("\nBoth required project datasets are available.")
print("Part A2 file verification completed successfully.")

TXT files found: 1
PDF files found: 1

Both required project datasets are available.
Part A2 file verification completed successfully.


## A3: Identify Data Type and Inspect Data



This section inspects the two project datasets to confirm that their content can be read successfully.

The selected sources are:

1. UTM FAI Lecturer Profiles (TXT)
2. UTM FAI FYP Handbook (PDF)

Both sources follow the document-based data preparation pathway (Pathway A).


### A3.1 Inspect lecturer TXT

In [7]:
# ============================================================
# CELL A3.1 — INSPECT LECTURER TXT
# Part A3: Read and inspect the lecturer profile dataset
# ============================================================

from pathlib import Path
import re

# Select the lecturer TXT file
txt_path = RAW_DIR / "fai_all_lecturer_profiles.txt"

# Confirm the file exists
assert txt_path.is_file(), f"File not found: {txt_path}"

# Read the original TXT content
txt_content = txt_path.read_text(encoding="utf-8")

# Identify lecturer record headings
record_ids = re.findall(
    r"^Record ID:\s*(FAI-\d+)",
    txt_content,
    flags=re.MULTILINE
)

print("=" * 55)
print("LECTURER TXT — INITIAL INSPECTION")
print("=" * 55)

print("Filename:", txt_path.name)
print("File size:", txt_path.stat().st_size, "bytes")
print("Total characters:", len(txt_content))
print("Lecturer records detected:", len(record_ids))

print("\n--- CONTENT PREVIEW ---")
print(txt_content[:1500])

assert txt_content.strip(), "TXT file contains no readable text."

print("\nTXT ingestion successful.")

LECTURER TXT — INITIAL INSPECTION
Filename: fai_all_lecturer_profiles.txt
File size: 34172 bytes
Total characters: 33335
Lecturer records detected: 117

--- CONTENT PREVIEW ---
UTM FAI LECTURER PROFILE DATASET
Collection date (UTC): 2026-11-17T18:26:42.789826+00:00
Source: Official UTM FAI department webpages
Status: Automatically extracted; manual verification required

---
Record ID: FAI-001
Name: Assoc. Prof. Dr. Nurulhuda Firdaus binti Mohd Azmi
Department: Intelligence Informatics
Research Expertise: Not provided
Source URL: https://fai.utm.my/department-of-intelligence-informatics-2/

---
Record ID: FAI-002
Name: Prof. Emeritus Datuk Dr. Osman bin Bakar
Department: Intelligence Informatics
Research Expertise: Not provided
Source URL: https://fai.utm.my/department-of-intelligence-informatics-2/

---
Record ID: FAI-003
Name: Prof. Dato’ Dr. Mohamad Fauzan bin Noordin
Department: Intelligence Informatics
Research Expertise: Not provided
Source URL: https://fai.utm.my/department-of-i

### A3.2 Inspect FYP Handbook PDF


The official FYP Handbook is inspected to determine its page count and confirm that readable text can be extracted. Representative page samples are displayed to identify possible extraction limitations before document preparation.


In [8]:
# ============================================================
# CELL A3.2 — INSPECT FYP HANDBOOK PDF
# Part A3: Check PDF readability and page content
# ============================================================

!pip -q install pypdf

from pypdf import PdfReader

# Find the uploaded PDF
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

assert len(pdf_files) == 1, (
    "Expected one FYP Handbook PDF. "
    f"Found: {[f.name for f in pdf_files]}"
)

pdf_path = pdf_files[0]

# Open the PDF
reader = PdfReader(str(pdf_path))

print("=" * 55)
print("FYP HANDBOOK PDF — INITIAL INSPECTION")
print("=" * 55)

print("Filename:", pdf_path.name)
print("File size:", pdf_path.stat().st_size, "bytes")
print("Total pages:", len(reader.pages))

# Inspect the first two pages
for page_index in range(min(2, len(reader.pages))):
    page_text = reader.pages[page_index].extract_text() or ""

    print(f"\n--- PAGE {page_index + 1} PREVIEW ---")
    print("Extracted characters:", len(page_text))
    print(page_text[:1000] if page_text.strip() else "[No readable text extracted]")

print("\nPDF inspection completed.")

FYP HANDBOOK PDF — INITIAL INSPECTION
Filename: utm_fai_fyp_guidelines.pdf.pdf
File size: 89523865 bytes
Total pages: 119

--- PAGE 1 PREVIEW ---
Extracted characters: 804
FINAL YEARFINAL YEARFINAL YEARFINAL YEAR
HANDBOOKHANDBOOK
import numpy as np
from ai.models import Model
def forward(x):
    W1, b1 = params["W1"], params["b1"]
    z1 = x @ W1 + b1
    a1 = relu(z1)
    W2, b2 = params["W2"], params["b2"]
    output = a1 @ W2 + b2
    return output
def relu(z):
    return np.maximum(0, z)
model = Model()
prediction = model.predict(data)
print("AI model executed successfully")
import numpy as np
from ai.models import Model
def forward(x):
    W1, b1 = params["W1"], params["b1"]
    z1 = x @ W1 + b1
    a1 = relu(z1)
    W2, b2 = params["W2"], params["b2"]
    output = a1 @ W2 + b2
    return output
def relu(z):
    return np.maximum(0, z)
model = Model()
prediction = model.predict(data)
print("AI model executed successfully")
PROJECT HANDBOOK
PROJECT
2026-2027

--- PAGE 2 PREVIEW ---

### A3.3 Summarize inspection evidence

In [9]:
# ============================================================
# CELL A3.3 — INSPECTION SUMMARY
# Part A3: Summarize both source documents
# ============================================================

print("=" * 55)
print("PART A3 — DATA INSPECTION SUMMARY")
print("=" * 55)

print("\nSOURCE 1: Lecturer Profiles")
print("Format: TXT")
print("Data type: Unstructured text")
print("Records detected:", len(record_ids))
print("Readable content:", bool(txt_content.strip()))

print("\nSOURCE 2: FYP Handbook")
print("Format: PDF")
print("Data type: Unstructured document")
print("Pages detected:", len(reader.pages))

# Count pages with extractable text
readable_pages = sum(
    bool((page.extract_text() or "").strip())
    for page in reader.pages
)

print("Pages with extractable text:", readable_pages)

print("\nInspection completed.")
print("Next step: A4 — Data Inventory")

PART A3 — DATA INSPECTION SUMMARY

SOURCE 1: Lecturer Profiles
Format: TXT
Data type: Unstructured text
Records detected: 117
Readable content: True

SOURCE 2: FYP Handbook
Format: PDF
Data type: Unstructured document
Pages detected: 119
Pages with extractable text: 115

Inspection completed.
Next step: A4 — Data Inventory



### A3.4 — Initial Data Inspection Observations

**Source 1: UTM FAI Lecturer Profiles (TXT)**

- The dataset contains lecturer profiles compiled from official UTM FAI department webpages.
- Each record is expected to contain a record ID, lecturer name, department, research expertise and source URL.
- Some lecturer profiles may contain missing research expertise.
- The text file requires further validation before it can be used for supervisor matching.

**Source 2: UTM FAI FYP Handbook (PDF)**

- The handbook contains official FYP-related information.
- The document is inspected for page count and extractable text.
- PDF extraction may introduce formatting, reading-order or table-structure issues.
- Page information should be preserved to support future citations.

**Initial Decision**

Pathway A (PDF/TXT) is selected for Part B because both datasets contain textual information.


## A4: Data Inventory



This section records the identity, purpose, access conditions, and initial inspection evidence for the two selected project datasets. The inventory supports source traceability and helps justify the data preparation decisions used in Part B.


### Identify the four pages without extractable text

In [10]:
# ============================================================
# CELL A4.1 — IDENTIFY PAGES WITHOUT EXTRACTABLE TEXT
# ============================================================

empty_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if not text.strip():
        empty_pages.append(page_number)

print("Total handbook pages:", len(reader.pages))
print("Pages without extractable text:", len(empty_pages))
print("Affected PDF page numbers:", empty_pages)

Total handbook pages: 119
Pages without extractable text: 4
Affected PDF page numbers: [10, 16, 18, 94]


### Create data inventory file

In [11]:
# ============================================================
# CELL A4.2 — CREATE DATA INVENTORY
# ============================================================

from pathlib import Path

DOCS_DIR = BASE_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

inventory = f"""# Week 3 — Data Inventory

## Project Information

**Project:** UTM FAI FYP & Supervisor Matching Assistant
**Course:** SAIA 3113 — AI Engineering
**Selected Pathway:** A — PDF/TXT Documents

## 1. Data Source Inventory

| Source ID | Filename / URL | Type and Purpose | Version / Date / Owner | Access / Rights | Inspection Evidence |
|---|---|---|---|---|---|
| S01 | `fai_all_lecturer_profiles.txt` | TXT; lecturer names, departments, and research expertise for supervisor matching | Compiled 17 November 2026; UTM FAI source webpages | Public faculty webpages; reuse and redistribution rights not independently confirmed | 117 records detected; 33,335 characters; readable TXT |
| S02 | `utm_fai_fyp_guidelines.pdf.pdf` | PDF; official FYP policies and procedures | Academic Session 2026/2027; UTM Faculty of Artificial Intelligence | Institutional handbook; redistribution permission not independently confirmed | 119 pages; 115 pages with extractable text; 4 pages without extractable text |

## 2. Requirements Traceability

- **FR1 — Supervisor Matching:** Requires lecturer names, departments, and research expertise from S01.
- **FR2 — FYP Policy Q&A:** Requires official handbook information from S02.
- **FR3 — Scope Refinement:** Uses relevant expertise and FYP guidance from both sources.
- **FR5 — Citations:** Requires preservation of source URLs and PDF page information.
- **NFR4 — Source Updates:** Requires source metadata and reproducible processing.

## 3. Initial Inspection Observations

### S01 — Lecturer Profiles

- File size: 34,172 bytes.
- Total characters: 33,335.
- Lecturer records detected: 117.
- The TXT contains labeled fields such as Record ID, Name, Department, Research Expertise, and Source URL.
- Some records have research expertise marked as Not provided.
- The dataset was compiled from official UTM FAI department webpages and manually reviewed.
- The TXT header should be updated to reflect its manual review status.

### S02 — FYP Handbook

- File size: 89,523,865 bytes.
- Total PDF pages: 119.
- Pages containing extractable text: 115.
- Pages without extractable text: 4.
- The first page contains repeated headings and duplicated code text in the extraction.
- Page-level metadata should be preserved for future citations.
- Pages without extractable text require further inspection before processing decisions are finalized.

## 4. Storage and Data Handling

**Raw data:** `data/raw/`

**Prepared data:** `data/processed/`

Original files will be retained separately from processed outputs.

The Colab workspace is temporary. Final approved artefacts will be saved to the existing project repository.

## 5. Selected Preparation Method

**Pathway A — PDF/TXT**

Planned processing includes document parsing, justified whitespace cleaning, text chunking, validation, and metadata preservation.

Missing lecturer expertise will not be invented. Supervisor availability will not be inferred from research expertise.
"""

inventory_path = DOCS_DIR / "week3_data_inventory.md"
inventory_path.write_text(inventory, encoding="utf-8")

print("Data inventory created successfully.")
print("Saved to:", inventory_path)
print("File size:", inventory_path.stat().st_size, "bytes")

Data inventory created successfully.
Saved to: /content/week3_project/docs/week3_data_inventory.md
File size: 2992 bytes


# Part B

## B1 — Prepare notebook for Part B


## B1. Document-Based Data Preparation

**Selected Pathway:** A — PDF/TXT Documents

**Input datasets:**
1. UTM FAI Lecturer Profiles (TXT)
2. UTM FAI FYP Handbook (PDF)

**Objective:** Prepare readable, traceable document content for the UTM FAI FYP & Supervisor Matching Assistant.

The original files are preserved in `data/raw/`. Processed outputs will be stored separately in `data/processed/`.

## B2. Common Metadata Helpers

### Cell 4 — Source Selection and Lineage Utilities

This cell defines reusable functions to:
- Select an exact source file.
- Calculate a SHA-256 file checksum.
- Save processing metadata in a JSON manifest.


In [12]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )

    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )

    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )

    return matches[0]


def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }

    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)

    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("Saved lineage:", target)
    return manifest


# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


## 4A: Common Lineage Registry


This cell initializes a lineage registry to track the relationship between original source files and their processed outputs.

The registry will record:
- Original source file path.
- Processed output file path.
- Description of the processing method.

This supports data traceability and reproducibility.

**Note:** Lineage registration will be performed after the selected Pathway A processing and B3 validation.


In [13]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways

from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )

    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )

    processing = processing.strip()

    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))

    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }

## 5A1: PDF/TXT Parsing, Cleaning and Chunking


**Selected Pathway:** A — Document Data

**First source:** `fai_all_lecturer_profiles.txt`

This step will:
1. Read text from the original document.
2. Remove unnecessary spaces and tabs.
3. Compare fixed-size and paragraph-aware chunking.
4. Inspect sample chunks before choosing a preparation method.

**Expected output:** Extraction statistics and two chunking comparisons.

**Note:** This step does not modify the original dataset.


In [14]:
# Cell 5A1 — PATHWAY A1: PDF/TXT
# CONDITIONAL REQUIRED: Document-based projects

!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
f = select_source(
    relative_path="fai_all_lecturer_profiles.txt"
)

# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")

# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."

# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]

# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")

    parts = []
    current = ""

    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()

        if not segment:
            continue

        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""

            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment

        else:
            if current:
                parts.append(current)
            current = segment

    if current:
        parts.append(current)

    return parts

# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: fai_all_lecturer_profiles.txt
Pages/segments: 1
Extracted characters: 33335
Cleaned characters: 33334
Content sample: 'UTM FAI LECTURER PROFILE DATASET\nCollection date (UTC): 2026-11-17T18:26:42.789826+00:00\nSource: Official UTM FAI department webpages\nStatus: Automatically extracted; manual verification required\n\n---\nRecord ID: FAI-001\nName: Assoc. Prof. Dr. Nurulhuda Firdaus binti Mohd Azmi\nDepartment: Intelligenc'

--- Chunking Comparison ---
Fixed chunks: 67
Paragraph-aware chunks: 101

Fixed sample:
'UTM FAI LECTURER PROFILE DATASET\nCollection date (UTC): 2026-11-17T18:26:42.789826+00:00\nSource: Official UTM FAI department webpages\nStatus: Automatically extracted; manual verifi'

Paragraph-aware sample:
'UTM FAI LECTURER PROFILE DATASET\nCollection date (UTC): 2026-11-17T18:26:42.789826+00:00\nSource: Official UTM FAI department webpages\nStatus: Automatically extracted; manual verifi'

PATHWAY A1 completed successfully.


In [16]:
# Additional Check — Lecturer Record Boundaries
# Verify whether chunking preserves lecturer records

import re

record_pattern = r"Record ID:\s*(FAI-\d+)"

original_ids = re.findall(record_pattern, clean)

print("Original lecturer records:", len(original_ids))
print("Unique lecturer IDs:", len(set(original_ids)))

print("\n--- Fixed-size Chunking ---")
fixed_record_counts = [
    len(re.findall(record_pattern, chunk))
    for chunk in fixed
]
print("Chunks:", len(fixed))
print("Record IDs found:", sum(fixed_record_counts))
print("Chunks containing multiple record IDs:",
      sum(count > 1 for count in fixed_record_counts))

print("\n--- Paragraph-aware Chunking ---")
structured_record_counts = [
    len(re.findall(record_pattern, chunk))
    for chunk in structured
]
print("Chunks:", len(structured))
print("Record IDs found:", sum(structured_record_counts))
print("Chunks containing multiple record IDs:",
      sum(count > 1 for count in structured_record_counts))

print("\n--- Sample Record Boundary ---")
for i, chunk in enumerate(structured):
    if re.search(record_pattern, chunk):
        print("Chunk number:", i + 1)
        print(chunk[:700])
        break

Original lecturer records: 117
Unique lecturer IDs: 117

--- Fixed-size Chunking ---
Chunks: 67
Record IDs found: 115
Chunks containing multiple record IDs: 47

--- Paragraph-aware Chunking ---
Chunks: 101
Record IDs found: 117
Chunks containing multiple record IDs: 16

--- Sample Record Boundary ---
Chunk number: 1
UTM FAI LECTURER PROFILE DATASET
Collection date (UTC): 2026-11-17T18:26:42.789826+00:00
Source: Official UTM FAI department webpages
Status: Automatically extracted; manual verification required

---
Record ID: FAI-001
Name: Assoc. Prof. Dr. Nurulhuda Firdaus binti Mohd Azmi
Department: Intelligence Informatics
Research Expertise: Not provided
Source URL: https://fai.utm.my/department-of-intelligence-informatics-2/


## 5A2: Export prepared text records



**Selected method:** Paragraph-aware chunking (500 characters)

**Justification:**
- All 117 lecturer record IDs were detected in the paragraph-aware chunks.
- Fixed-size chunking detected only 115 of the 117 record IDs.
- Paragraph-aware chunking better respects text boundaries.
- However, 16 chunks contain multiple lecturer IDs, so further record-aware preparation may be needed.

**Output:** `data/processed/prepared_text.jsonl`

Each exported record will include the chunk text, source filename, source checksum, processing method, and a record ID.


In [17]:
# Cell 5A2 — PATHWAY A2
# REQUIRED after A1
# Choose and export traceable text chunks

import json

# Step 1: Select the preparation approach
CHOSEN_METHOD = "paragraph-aware"

if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")

# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]

# Page is None because the basic whole-document
# splitter does not preserve page boundaries.

# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)

with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"

assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 101
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 58412 bytes
Validation: PASS
First record: {'record_id': 'fai_all_lecturer_profiles-0001', 'text': 'UTM FAI LECTURER PROFILE DATASET\nCollection date (UTC): 2026-11-17T18:26:42.789826+00:00\nSource: Official UTM FAI department webpages\nStatus: Automatically extracted; manual verification required\n\n---\nRecord ID: FAI-001\nName: Assoc. Prof. Dr. Nurulhuda Firdaus binti Mohd Azmi\nDepartment: Intelligence Informatics\nResearch Expertise: Not provided\nSource URL: https://fai.utm.my/department-of-intelligence-informatics-2/', 'source': 'fai_all_lecturer_profiles.txt', 'source_sha256': '7ecc0ab847de8020ac6e4891b8038c029c96ebf81f95f05e009cc6e0159626bf', 'processing': 'paragraph-aware-char-500', 'page': None}


## B3 — Validate the prepared dataset



**Source:** UTM FAI Lecturer Profiles (TXT)

**Comparison:**
- Fixed-size chunking: 67 chunks; 115 lecturer IDs detected.
- Paragraph-aware chunking: 101 chunks; 117 lecturer IDs detected.

**Selected approach:** Paragraph-aware chunking (500 characters).

**Benefit:** All 117 lecturer IDs remain detectable, and paragraph boundaries are better preserved.

**Accepted limitation:** Some chunks contain multiple lecturer records, and the method does not guarantee that every lecturer profile remains complete within one chunk.

**Validation objective:** Confirm that the prepared JSONL file exists, is non-empty, and contains readable records with source metadata.


In [18]:
# Step B3 — Basic Output Contract
# REQUIRED after selected Pathway A

assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'

print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 58412
Validation: file exists and is non-empty; verify content manually too.


In [19]:
# Additional B3 Check — Inspect prepared JSONL content

import json

with prepared_path.open("r", encoding="utf-8") as file:
    checked_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

required_fields = {
    "record_id", "text", "source",
    "source_sha256", "processing", "page"
}

assert len(checked_records) == 101, "Unexpected chunk count"
assert all(required_fields.issubset(row) for row in checked_records)
assert all(row["text"].strip() for row in checked_records)
assert all(
    row["source"] == "fai_all_lecturer_profiles.txt"
    for row in checked_records
)

print("Total exported records:", len(checked_records))
print("All required metadata fields present: PASS")
print("All records contain non-empty text: PASS")
print("Source identity: PASS")

print("\n--- Representative Record ---")
sample = checked_records[0]
print("Record ID:", sample["record_id"])
print("Source:", sample["source"])
print("Processing:", sample["processing"])
print("Text preview:", repr(sample["text"][:300]))

Total exported records: 101
All required metadata fields present: PASS
All records contain non-empty text: PASS
Source identity: PASS

--- Representative Record ---
Record ID: fai_all_lecturer_profiles-0001
Source: fai_all_lecturer_profiles.txt
Processing: paragraph-aware-char-500
Text preview: 'UTM FAI LECTURER PROFILE DATASET\nCollection date (UTC): 2026-11-17T18:26:42.789826+00:00\nSource: Official UTM FAI department webpages\nStatus: Automatically extracted; manual verification required\n\n---\nRecord ID: FAI-001\nName: Assoc. Prof. Dr. Nurulhuda Firdaus binti Mohd Azmi\nDepartment: Intelligenc'


## B4.1 — Register Lineage



**Original source:** `data/raw/fai_all_lecturer_profiles.txt`

**Prepared output:** `data/processed/prepared_text.jsonl`

**Processing method:** Paragraph-aware chunking (500 characters)

**Purpose:** Register the relationship between the original lecturer dataset and its processed output.

This allows us to trace prepared data back to its original source and the preparation method used.

**Note:** The common lineage manifest will be generated after all selected sources have been processed and registered.


In [21]:
# Step B4.1 — Register Pathway A (PDF/TXT)
# Lecturer Profiles Dataset

register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

In [22]:
# Additional Check — Verify registered lineage

print("Registered lineage entries:", len(lineage_entries))

for entry in lineage_entries.values():
    print("Source:", entry["source"].name)
    print("Output:", entry["output"].name)
    print("Processing:", entry["processing"])

Registered lineage entries: 1
Source: fai_all_lecturer_profiles.txt
Output: prepared_text.jsonl
Processing: paragraph-aware-char-500


## 5A1 for the FYP Handbook PDF



**Selected source:** `utm_fai_fyp_guidelines.pdf.pdf`

**Purpose:**
1. Extract readable text from the FYP Handbook.
2. Inspect extraction statistics.
3. Apply basic whitespace cleaning.
4. Compare fixed-size and paragraph-aware chunking.
5. Identify any extraction limitations.

**Known limitation from Part A:** The PDF contains 119 pages, of which 115 had extractable text. Pages 10, 16, 18 and 94 had no extractable text using `pypdf`.

**Note:** The original PDF will remain unchanged.

In [23]:
# Cell 5A1 — PATHWAY A1: FYP Handbook PDF
# Adapted from the lecturer's Pathway A1 for the actual PDF source

from pypdf import PdfReader
import re

# Step 1: Select the exact PDF
f = select_source(
    relative_path="utm_fai_fyp_guidelines.pdf.pdf"
)

# Step 2: Extract text from each PDF page
reader = PdfReader(str(f))

pages = [
    (i + 1, page.extract_text() or "")
    for i, page in enumerate(reader.pages)
]

# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."

# Step 4: Reuse the lecturer's chunking functions from Cell 5A1
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

# Step 5: Compare the approaches
print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

# Step 6: Check pages without extractable text
empty_pages = [
    page_number
    for page_number, content in pages
    if not content.strip()
]

print("\n--- PDF Extraction Quality ---")
print("Total pages:", len(pages))
print("Pages with extractable text:", len(pages) - len(empty_pages))
print("Pages without extractable text:", empty_pages)

print("\nPATHWAY A1 completed successfully.")

Source: utm_fai_fyp_guidelines.pdf.pdf
Pages/segments: 119
Extracted characters: 187647
Cleaned characters: 186132
Content sample: 'FINAL YEARFINAL YEARFINAL YEARFINAL YEAR\nHANDBOOKHANDBOOK\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x @ W1 + b1\n a1 = relu(z1)\n W2, b2 = params["W2"], params["b2"]\n output = a1 @ W2 + b2\n return output\ndef relu(z):\n return np.maximum(0,'

--- Chunking Comparison ---
Fixed chunks: 373
Paragraph-aware chunks: 438

Fixed sample:
'FINAL YEARFINAL YEARFINAL YEARFINAL YEAR\nHANDBOOKHANDBOOK\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x @ W1 + b1\n a1 '

Paragraph-aware sample:
'FINAL YEARFINAL YEARFINAL YEARFINAL YEAR\nHANDBOOKHANDBOOK\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x @ W1 + b1\n a1 '

--- PDF Extraction Quality ---
Total pages: 119
Pages with

## 5A2: Export FYP Handbook records



**Source:** `utm_fai_fyp_guidelines.pdf.pdf`

**Selected method:** Paragraph-aware chunking (500 characters)

**Comparison:**
- Fixed-size: 373 chunks
- Paragraph-aware: 438 chunks

**Justification:** Paragraph-aware chunking attempts to preserve meaningful text boundaries for retrieving FYP guidelines.

**Known limitations:**
- Some extracted text contains repeated headings or layout artifacts.
- Pages 10, 16, 18 and 94 have no extractable text.
- Whole-document chunking does not preserve exact PDF page references.

**Output:** `data/processed/prepared_fyp_handbook.jsonl`

A separate filename is used to prevent overwriting the prepared lecturer dataset.


In [24]:
# Cell 5A2 — PATHWAY A2: FYP Handbook PDF
# Adapted from lecturer's Cell 5A2
# Modification: Separate output filename for this source

import json

# Step 1: Select preparation approach
CHOSEN_METHOD = "paragraph-aware"

if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")

# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]

# Page is None because whole-document chunking
# does not preserve exact page boundaries.

# Step 3: Export to a separate JSONL file
prepared_path = PROCESSED_DIR / "prepared_fyp_handbook.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)

with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

# Step 4: Validate exported file
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"

assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 438
Prepared file: /content/week3_project/data/processed/prepared_fyp_handbook.jsonl
File size: 304714 bytes
Validation: PASS
First record: {'record_id': 'utm_fai_fyp_guidelines.pdf-0001', 'text': 'FINAL YEARFINAL YEARFINAL YEARFINAL YEAR\nHANDBOOKHANDBOOK\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x @ W1 + b1\n a1 = relu(z1)\n W2, b2 = params["W2"], params["b2"]\n output = a1 @ W2 + b2\n return output\ndef relu(z):\n return np.maximum(0, z)\nmodel = Model()\nprediction = model.predict(data)\nprint("AI model executed successfully")\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x', 'source': 'utm_fai_fyp_guidelines.pdf.pdf', 'source_sha256': '4b698d76fcb230f5d15a817b3b6949c859ccf96f7aeb9e9fab708dbe9d4d6998', 'processing': 'paragraph-aware-char-500', 'page'

## B3: Validate the FYP Handbook PDF


**Source:** `utm_fai_fyp_guidelines.pdf.pdf`

**Prepared output:** `prepared_fyp_handbook.jsonl`

**Chunking comparison:**
- Fixed-size chunking: 373 chunks
- Paragraph-aware chunking: 438 chunks

**Selected method:** Paragraph-aware chunking (500 characters).

**Benefit:** Attempts to preserve paragraph boundaries, making related FYP guidelines easier to retrieve together.

**Accepted limitations:**
- Repeated headings and other PDF extraction artifacts are present.
- Four pages have no extractable text.
- Exact page references are not preserved in the exported chunks.

**Validation objective:** Confirm that the prepared JSONL exists, is non-empty, and contains valid records with source metadata.


In [25]:
# Step B3 — Basic Output Contract
# FYP Handbook PDF

assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'

print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_fyp_handbook.jsonl Size: 304714
Validation: file exists and is non-empty; verify content manually too.


In [26]:
# Additional B3 Check — FYP Handbook PDF
# Verify exported JSONL content and metadata

import json

with prepared_path.open("r", encoding="utf-8") as file:
    checked_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

required_fields = {
    "record_id",
    "text",
    "source",
    "source_sha256",
    "processing",
    "page"
}

assert len(checked_records) == 438, "Unexpected chunk count"

assert all(
    required_fields.issubset(row)
    for row in checked_records
), "Missing required metadata"

assert all(
    row["text"].strip()
    for row in checked_records
), "Empty text chunk detected"

assert all(
    row["source"] == "utm_fai_fyp_guidelines.pdf.pdf"
    for row in checked_records
), "Incorrect source identity"

print("Total exported records:", len(checked_records))
print("All required metadata fields present: PASS")
print("All records contain non-empty text: PASS")
print("Source identity: PASS")

print("\n--- Representative Record ---")
sample = checked_records[0]

print("Record ID:", sample["record_id"])
print("Source:", sample["source"])
print("Processing:", sample["processing"])
print("Text preview:", repr(sample["text"][:300]))

Total exported records: 438
All required metadata fields present: PASS
All records contain non-empty text: PASS
Source identity: PASS

--- Representative Record ---
Record ID: utm_fai_fyp_guidelines.pdf-0001
Source: utm_fai_fyp_guidelines.pdf.pdf
Processing: paragraph-aware-char-500
Text preview: 'FINAL YEARFINAL YEARFINAL YEARFINAL YEAR\nHANDBOOKHANDBOOK\nimport numpy as np\nfrom ai.models import Model\ndef forward(x):\n W1, b1 = params["W1"], params["b1"]\n z1 = x @ W1 + b1\n a1 = relu(z1)\n W2, b2 = params["W2"], params["b2"]\n output = a1 @ W2 + b2\n return output\ndef relu(z):\n return np.maximum(0,'


## B4.1: Register FYP Handbook PDF Lineage



**Original source:** `data/raw/utm_fai_fyp_guidelines.pdf.pdf`

**Prepared output:** `data/processed/prepared_fyp_handbook.jsonl`

**Processing method:** Paragraph-aware chunking (500 characters)

**Purpose:** Register the relationship between the original FYP Handbook PDF and its processed JSONL output.

**Known limitations:**
- Four PDF pages have no extractable text.
- Some extracted text contains layout artifacts.
- Exact PDF page references are not preserved.

This registration will be combined with the lecturer-profile lineage when generating the common manifest.


In [27]:
# Step B4.1 — Register Pathway A (PDF/TXT)
# FYP Handbook PDF

register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

In [28]:
# Additional Check — Verify both registered sources

print("Registered lineage entries:", len(lineage_entries))

for entry in lineage_entries.values():
    print("\nSource:", entry["source"].name)
    print("Output:", entry["output"].name)
    print("Processing:", entry["processing"])

assert len(lineage_entries) == 2, \
    "Expected two registered source-to-output relationships"

print("\nB4.1 Lineage Verification: PASS")

Registered lineage entries: 2

Source: fai_all_lecturer_profiles.txt
Output: prepared_text.jsonl
Processing: paragraph-aware-char-500

Source: utm_fai_fyp_guidelines.pdf.pdf
Output: prepared_fyp_handbook.jsonl
Processing: paragraph-aware-char-500

B4.1 Lineage Verification: PASS


## 6A: Generate the Common Lineage Manifest



**Purpose:** Generate one manifest containing both prepared datasets and their source-to-output relationships.

**Registered sources:**
1. UTM FAI Lecturer Profiles (TXT)
2. UTM FAI FYP Handbook (PDF)

**Expected output:** `data/week3_manifest.json`

The manifest records:
- Source filename and SHA-256 checksum
- Prepared output location and SHA-256 checksum
- Processing method
- Source version and access-rights verification status
- Data preparation status

**Known limitations:** Some lecturer records may be split or grouped across chunks. The PDF contains extraction artifacts, four pages without extractable text, and no page-level references in the prepared JSONL. Data quality and permission to share still require review.


In [29]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered

assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


## 6B: Verify the Common Lineage Manifest



**Purpose:** Verify that the generated `week3_manifest.json` is valid and contains correct source-to-output relationships.

**Validation checks:**
1. Confirm the manifest file exists.
2. Confirm the manifest contains lineage records.
3. Check that the number of records matches the lineage registry.
4. Verify that all original source files exist.
5. Verify that all prepared output files exist.
6. Compare SHA-256 checksums to detect file changes.

**Expected result:** Two verified lineage records.

**Note:** This validation confirms file integrity and traceability. It does not prove that the extracted content is completely accurate or that the data is authorised for public sharing.

In [30]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A

import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"

assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])

assert saved_records, (
    "No lineage records found in saved manifest."
)

assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"

    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )

    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


# Part C — Build and Document the V1 Data Pipeline


#### Cell 7 — Common V1 Pipeline Utilities

**Purpose:** Create reusable functions for running a complete data preparation pipeline.

The pipeline will follow these stages:

1. **Ingest:** Identify the original dataset.
2. **Inspect:** Check the source content.
3. **Prepare and Store:** Clean, chunk, and save the data.
4. **Validate:** Confirm that output files exist and are non-empty.
5. **Record Lineage:** Save source information, processing configuration, checksums, and output details.

**Selected pathway:** Cell 8A — PDF/TXT

**Project datasets:**
- `fai_all_lecturer_profiles.txt`
- `utm_fai_fyp_guidelines.pdf.pdf`

**Preparation configuration:** Paragraph-aware chunking, 500 characters.

**Expected result:** Cell 7 utility functions are ready. No dataset is processed yet.


In [31]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record

# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }

    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


## 8A — PDF/TXT Document Pipeline

In [32]:
# Cell 8A — PDF/TXT Document Pipeline
# Adapted from lecturer's Cell 8A for UTM FAI data

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select source and configuration
SOURCE_A = resolve_source(
    "fai_all_lecturer_profiles.txt"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]

    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }

# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)

print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


## 8A for FYP Handbook PDF


**Source:** `utm_fai_fyp_guidelines.pdf.pdf`

**Selected configuration:**
- Method: Paragraph-aware chunking
- Chunk size: 500 characters

**Purpose:** Execute the same V1 document pipeline for the FYP Handbook PDF.

**Project-specific adaptation:** Use a separate pipeline identifier, `documents_fyp_handbook`, to prevent overwriting the lecturer-profile pipeline.

**Expected outputs:**
- `data/pipeline_v1/documents_fyp_handbook/prepared_text.jsonl`
- `data/pipeline_v1/documents_fyp_handbook/pipeline_manifest.json`

**Known limitations:** PDF extraction artifacts, four pages without extractable text, and missing page-level references.


In [33]:
# Cell 8A — FYP Handbook PDF
# Project-specific adaptation for a second document source

# Step 1: Select the FYP Handbook PDF
SOURCE_A_PDF = resolve_source(
    "utm_fai_fyp_guidelines.pdf.pdf"
)

# Step 2: Use the same configuration as Part B
CONFIG_A_PDF = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 3: Execute the PDF pipeline
# Reuse inspect_documents() and prepare_documents()
# already defined in the previous Cell 8A.

run_A_PDF = register_and_run(
    "documents_fyp_handbook",
    SOURCE_A_PDF,
    inspect_documents,
    prepare_documents,
    CONFIG_A_PDF
)

print("FYP Handbook PDF pipeline completed successfully.")

SUCCESS: documents_fyp_handbook | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_fyp_handbook/pipeline_manifest.json
FYP Handbook PDF pipeline completed successfully.


## C9: Repeatability and Provenance Verification



**Purpose:** Check whether the V1 data pipelines produce consistent results when executed again.

**Registered pipelines:**
1. `documents` — Lecturer Profiles (TXT)
2. `documents_fyp_handbook` — FYP Handbook (PDF)

**Verification checks:**
- Original source SHA-256 fingerprints match.
- Prepared output SHA-256 fingerprints match.
- Prepared output sizes and counts match.
- Source-to-output mappings remain unchanged.

**Expected result:** Both pipelines pass the repeatability smoke test.

**Limitation:** This test verifies repeatability within the current Colab environment. It does not prove full reproducibility across different computers or guarantee data quality.

In [34]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_fyp_handbook | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_fyp_handbook/pipeline_manifest.json
PASS: documents_fyp_handbook — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
